In [18]:
import random
from numpy import *
from functools import reduce

def sigmoid(inX):
    return 1.0/(1+exp(-inX))

In [19]:
#节点类，负责记录和维护节点自身信息以及与这个节点相关的上下游连接，实现输出值和误差项的计算
class Node(object):
    def __init__(self,layer_index,node_index):
        """构造节点对象。layer_index节点所属的层的编号。node_index节点的编号。"""
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.upstream = []
        self.output = 0
        self.delta = 0

    def set_output(self,output):
        """设置节点的输出值，如果节点属于输入层会用到这个函数。"""
        self.output = output

    def append_downstream_connection(self,conn):
        """添加一个到下游节点的连接"""
        self.downstream.append(conn)

    def append_upstream_connection(self,conn):
        """添加一个到上游节点的连接"""
        self.upstream.append(conn)

    def calc_output(self):
        """计算节点的输出"""
        output = reduce(lambda ret,conn:ret+conn.upstream_node.output*conn.weight,self.upstream,0)
        self.output = sigmoid(output)

    def calc_hidden_layer_delta(self):
        """节点属于隐藏层时，计算delta"""
        downstream_delta = reduce(lambda ret,conn:ret+conn.downstream_node.delta*conn.weight,self.downstream,0.0)
        self.delta = self.output*(1-self.output)*downstream_delta

    def calc_output_layer_delta(self,label):
        """节点属于输出层时，计算delta"""
        self.delta = self.output*(1-self.output)*(label-self.output)

    def __str__(self):
        """打印节点信息"""
        node_str = '%u-%u: output: %f delta: %f' % (self.layer_index, self.node_index, self.output, self.delta)
        downstream_str = reduce(lambda ret, conn: ret + '\n\t' + str(conn), self.downstream, '')
        upstream_str = reduce(lambda ret, conn:ret + '\n\t' + str(conn), self.upstream, '')
        return node_str + '\n\tdownstream:' + downstream_str + '\n\tupstream:' + upstream_str

In [20]:
#Layer对象，负责初始化一层。此外，作为Node的集合对象，提供对Node集合的操作
class Layer(object):
    def __init__(self, layer_index, node_count):
        """初始化一层。layer_index层编号。node_count层所包含的节点个数"""
        self.layer_index = layer_index
        self.nodes = []
        for i in range(node_count): # 让self.nodes这个空列表中按节点数量放满节点
            self.nodes.append(Node(layer_index,i))
        self.nodes.append(ConstNode(layer_index,node_count))

    def set_output(self, data):
        """设置层的输出。当层是输入层时会用到"""
        for i in range(len(data)):
            self.nodes[i].set_output(data[i])

    def calc_output(self):
        """计算层的输出向量"""
        for node in self.nodes[:-1]:
            node.calc_output()

    def dump(self):
        """打印层的信息"""
        for node in self.nodes:
            print(node)

In [21]:
#Connection对象，主要职责是记录连接的权重，以及这个连接所关联的上下游节点
class Connection(object):
    def __init__(self,upstream_node,downstream_node):
        """初始化连接，权重初始化为一个很小的随机数。upstream_node连接的上游节点。downstream_node连接的下游节点。"""
        self.upstream_node = upstream_node
        self.downstream_node = downstream_node
        self.weight = random.uniform(-0.1,0.1)
        self.gradient = 0.0

    def calc_gradient(self):
        """计算梯度"""
        self.gradient = self.downstream_node.delta*self.upstream_node.output
    
    def getgradient(self):
        """获取当前梯度"""
        return self.gradient
    
    def update_weight(self,rate): 
        """根据梯度下降法更新权重"""
        self.calc_gradient()
        self.weight +=rate*self.gradient

    def __str__(self):
        """打印连接信息"""
        return '(%u-%u) -> (%u -%u) = %f' % (self.upstream_node.layer_index,self.upstream_node.node_index,self.downstream_node.layer_index,self.downstream_node.node_index,self.weight)

In [22]:
#Connect对象，提供Connection集合操作
class Connections(object):
    def __init__(self):
        self.connections = []

    def add_connection(self, connection):
        self.connections.append(connection)
    
    def dump(self):
        for conn in self.connections:
            print(conn)

In [23]:
#Network对象,提供API
class Network(object):
    def __init__(self,layers):
        """初始化一个全连接神经网络。layers二维数组，描述神经网络每层节点数。"""
        self.connections = Connections()
        self.layers = []
        layer_count = len(layers)
        node_count = 0
        for i in range(layer_count):
            self.layers.append(Layer(i,layers[i]))  # Layer(i,layers[i]) 创建层对象，层编号是i，层节点数量是layers[i]。
        for layer in range(layer_count-1):
            connections = [Connection(upstream_node,downstream_node) for upstream_node in self.layers[layer].nodes for downstream_node in self.layers[layer + 1].nodes[:-1]]
            for conn in connections:
                self.connections.add_connection(conn)
                conn.downstream_node.append_upstream_connection(conn)
                conn.upstream_node.append_downstream_connection(conn)

    def train(self,labels,data_set,rate,iteration):
        """训练神经网络。labels数组，训练样板标签。data_set二维数组，训练样本特征，每个元素是一个样本的特征。"""
        for i in range(iteration):
            for d in range(len(data_set)):
                self.train_one_sample(labels[d],data_set[d],rate)
                
    def train_one_sample(self,label,sample,rate):
        """内部函数。用一个样本训练网络"""
        self.predict(sample)  #就是data_set[d]
        self.calc_delta(label)  #就是labels[d]
        self.update_weight(rate)
    def calc_delta(self,label):
        """内部函数。计算每个节点的delta"""
        output_nodes = self.layers[-1].nodes #列表。最后一层的所有节点。
        for i in range(len(label)): #label参数实际上预期是一个列表（或数组），长度等于输出层非偏置节点的个数（即神经网络的输出维度）
            output_nodes[i].calc_output_layer_delta(label[i]) #计算输出层delta 
        for layer in self.layers[-2::-1]: #排除输出层。从后往前反向传播，计算每一层delt
            for node in layer.nodes:
                node.calc_hidden_layer_delta()
    def update_weight(self,rate):
        """内部函数。更新每个连接权重"""
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.update_weight(rate)
    def calc_gradient(self):
        """内部函数。计算每个连接的梯度"""
        for layer in self.layers[:-1]: #遍历除最后一层外的每层
            for node in layer.nodes: #遍历该层所有节点
                for conn in node.downstream: # 遍历该节点的所有下游连接
                    conn.calc_gradient() # 所有连接计算梯度
    def get_gradient(self,label,sample):
        """获得网络在下一个样本下，每个连接上的梯度。label样本标签。sample样本输入。"""
        self.predict(sample)
        self.calc_delta(label)
        self.calc_gradient()
    def predict(self,sample):
        """根据输入的样本预测输出值。sample数组，样本的特征，也就是网络的输入向量。看Layer类来理解。"""
        self.layers[0].set_output(sample) # 计算输入层（第1层）的输出
        for i in range(1,len(self.layers)): # 依次往后面的层进行计算。看Layer类来理解。最后算出这一个sample经过整个神经网络后的输出。
            self.layers[i].calc_output()
        return map(lambda node:node.output,self.layers[-1].nodes[:-1]) # self.layers[-1].nodes[:-1]：获取输出层的所有节点，但排除最后一个偏置节点。将所有返回的output值收集起来，返回一个迭代器。

    def dump(self):
        """打印网络信息"""
        for layer in self.layers:
            layer.dump()

In [24]:
# 将一个 0~255 之间的整数（或更小的整数，因为只有 8 位掩码）转换为一个长度为 8 的向量
# 编码时用 0.9 表示该位为 1，用 0.1 表示该位为 0；解码时根据向量分量是否大于 0.5 还原出二进制位，再累加得到原始数字
# 常用于神经网络的输出层（或输入层），当类别数量不多（如 8 个以下）时，可以用这种二进制表示替代 one‑hot 编码，从而降低输出维度（8 维可以表示 256 个类别，而 one‑hot 需要 256 维）
class Normalizer(object):
    def __init__(self):
        self.mask = [0x1, 0x2, 0x4, 0x8, 0x10, 0x20, 0x40, 0x80] #用16进制表示的：1,2,4,8,16,32,64,128,256

    def norm(self, number): #将输入数字按位拆分为8个值，每个值对应0.9（位为1）或0.1（位为0）
        return [0.9 if number & m else 0.1 for m in self.mask]

    def denorm(self, vec):
        """将1个8维向量（每个元素是接近0.1或0.9的浮点数）还原为原始整数。"""
        binary = [1 if i > 0.5 else 0 for i in vec] #通过阈值0.5二值化，然后乘以对应权重，最后用reduce求和
        for i in range(len(self.mask)):
            binary[i] = binary[i] * self.mask[i]
        return reduce(lambda x,y: x + y, binary)


def mean_square_error(vec1, vec2):
    """计算半均方根误差，衡量网络输出vec1与目标向量vec2之间的差异"""
    return 0.5 * reduce(lambda a, b: a + b, map(lambda v: (v[0] - v[1]) * (v[0] - v[1]), zip(vec1, vec2)))


def gradient_check(network, sample_feature, sample_label):
    '''
    梯度检查
    network: 神经网络对象
    sample_feature: 样本的特征
    sample_label: 样本的标签
    '''
    # 计算网络误差
    network_error = lambda vec1, vec2: 0.5 * reduce(lambda a, b: a + b,map(lambda v: (v[0] - v[1]) * (v[0] - v[1]),zip(vec1, vec2)))

    # 获取网络在当前样本下每个连接的梯度
    network.get_gradient(sample_feature, sample_label)

    # 对每个权重做梯度检查    
    for conn in network.connections.connections: 
        actual_gradient = conn.get_gradient() # 获取指定连接的梯度
        # 增加一个很小的值，计算网络的误差
        epsilon = 0.0001
        conn.weight += epsilon
        error1 = network_error(network.predict(sample_feature), sample_label)
        # 减去一个很小的值，计算网络的误差
        conn.weight -= 2 * epsilon # 刚才加过了一次，因此这里需要减去2倍
        error2 = network_error(network.predict(sample_feature), sample_label)
        # 根据式6计算期望的梯度值
        expected_gradient = (error2 - error1) / (2 * epsilon)
        # 打印
        print('expected gradient: \t%f\nactual gradient: \t%f' % (expected_gradient, actual_gradient))


def train_data_set():
    normalizer = Normalizer()
    data_set = []
    labels = []
    for i in range(0, 256, 8):
        n = normalizer.norm(int(random.uniform(0, 256)))
        data_set.append(n)
        labels.append(n)
    return labels, data_set

def train(network):
    labels, data_set = train_data_set()
    network.train(labels, data_set, 0.3, 50)

def test(network, data):
    normalizer = Normalizer()
    norm_data = normalizer.norm(data)
    predict_data = network.predict(norm_data)
    print('\ttestdata(%u)\tpredict(%u)' % (data, normalizer.denorm(predict_data)))

def correct_ratio(network):
    normalizer = Normalizer()
    correct = 0.0;
    for i in range(256):
        if normalizer.denorm(network.predict(normalizer.norm(i))) == i:
            correct += 1.0
    print('correct_ratio: %.2f%%' % (correct / 256 * 100))

def gradient_check_test():
    net = Network([2, 2, 2])
    sample_feature = [0.9, 0.1]
    sample_label = [0.9, 0.1]
    gradient_check(net, sample_feature, sample_label)

if __name__ == '__main__':
    net = Network([8, 3, 8])
    train(net)
    net.dump()
    correct_ratio(net)

0-0: output: 0.900000 delta: 0.002519
	downstream:
	(0-0) -> (1 -0) = 0.148145
	(0-0) -> (1 -1) = -1.334202
	(0-0) -> (1 -2) = -2.219058
	upstream:
0-1: output: 0.100000 delta: 0.000731
	downstream:
	(0-1) -> (1 -0) = 0.798835
	(0-1) -> (1 -1) = -2.238470
	(0-1) -> (1 -2) = 0.036072
	upstream:
0-2: output: 0.100000 delta: 0.000960
	downstream:
	(0-2) -> (1 -0) = 0.929470
	(0-2) -> (1 -1) = -0.679413
	(0-2) -> (1 -2) = 1.311207
	upstream:
0-3: output: 0.900000 delta: -0.007855
	downstream:
	(0-3) -> (1 -0) = -2.127681
	(0-3) -> (1 -1) = 0.601463
	(0-3) -> (1 -2) = -0.034168
	upstream:
0-4: output: 0.100000 delta: -0.001801
	downstream:
	(0-4) -> (1 -0) = -0.554882
	(0-4) -> (1 -1) = 0.033947
	(0-4) -> (1 -2) = -0.249507
	upstream:
0-5: output: 0.900000 delta: 0.001806
	downstream:
	(0-5) -> (1 -0) = 0.763529
	(0-5) -> (1 -1) = 0.860293
	(0-5) -> (1 -2) = 1.367808
	upstream:
0-6: output: 0.900000 delta: 0.005849
	downstream:
	(0-6) -> (1 -0) = 0.669569
	(0-6) -> (1 -1) = 1.203877
	(0-6) 